# Step 1 — Data Engineering & K-Means Clustering
## Detecting Primary (RP) vs Secondary (RS) Residences

---

**Dataset:** Enedis Open Data · RES2-6-9kVA · Residential, electric heating, 6–9 kVA subscribed power  
**Raw format:** `valeur` = energy per 30-min slot in **Wh** (Watt-hours)  
**Coverage:** 500 customers · 364 days (2023-11-01 → 2024-10-29) · 17 472 readings / customer

### Physics Rule (strictly applied)
$$E_{kWh} = P_{kW} \times 0.5\ h \quad\equiv\quad \frac{E_{Wh}}{1000}$$

The Enedis `valeur` column already stores **integrated energy** (Wh) per 30-min slot.  
Dividing by 1000 converts to kWh — which is exactly the professor's formula applied to the
average slot power.

---

### Pipeline
| Step | Description |
|------|-------------|
| 1 | Load & convert to kWh |
| 2 | Parse temporal features (slot, season, weekend) |
| 3 | Build daily profile matrix (500 × 48) |
| 4 | Engineer statistical + temporal aggregation features |
| 5 | **Fourier decomposition** — intra-day profile shape & inter-day periodicity |
| 6 | **PCA** — dimensionality reduction before clustering |
| 7 | **K-Means (k=2)** — unsupervised RP / RS labelling |
| 8 | Plotly validation — average curves per cluster |

In [ ]:
# ── Imports & project-root setup ──────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from numpy.fft import rfft
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Navigate 2 levels up from  notebooks/01_clustering/  →  project root
PROJECT_ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.back_end.config.settings import (
    DATA_PROCESSED_DIR,
    METER_ID_COL,
    SUMMER_MONTHS,
    TIMESTAMP_COL,
    VALUE_COL,
    WINTER_MONTHS,
)
from src.back_end.utils.data_loader import load_raw_data

print(f'Project root : {PROJECT_ROOT}')
print('All imports  : OK')

---
## 1 · Load Raw Data & Apply the Physics Rule

In [ ]:
print('Loading raw data … (~8.7 M rows, may take 10–20 s)')
df = load_raw_data()

# ── Physics rule: Wh → kWh ──────────────────────────────────────────────────
# valeur is in Wh (confirmed in settings.py comment).
# Professor's formula:  E_kWh = P_kW × 0.5 h
# With valeur in Wh:    E_kWh = valeur / 1000
# Both are identical:   (valeur_Wh / 500 kW) × 0.5 h = valeur / 1000 kWh  ✓
df['energy_kwh'] = df[VALUE_COL] / 1000.0

n_customers = df[METER_ID_COL].nunique()
print(f'\nRows       : {len(df):>10,}')
print(f'Customers  : {n_customers:>10,}')
print(f'Date range : {df[TIMESTAMP_COL].min().date()}  →  {df[TIMESTAMP_COL].max().date()}')
print(f'\nenergy_kwh per slot (30 min):')
display(df['energy_kwh'].describe().rename('energy_kwh').to_frame().T.round(4))

---
## 2 · Temporal Feature Parsing

In [ ]:
# Slot index 0–47  (0 = 00:00, 1 = 00:30, …, 47 = 23:30)
df['slot']       = df[TIMESTAMP_COL].dt.hour * 2 + df[TIMESTAMP_COL].dt.minute // 30
df['hour']       = df[TIMESTAMP_COL].dt.hour
df['date']       = df[TIMESTAMP_COL].dt.date
df['dayofweek']  = df[TIMESTAMP_COL].dt.dayofweek    # 0 = Monday
df['month']      = df[TIMESTAMP_COL].dt.month
df['is_weekend'] = df['dayofweek'].isin([5, 6])       # Saturday or Sunday
df['is_winter']  = df['month'].isin(WINTER_MONTHS)    # Nov, Dec, Jan, Feb, Mar
df['is_summer']  = df['month'].isin(SUMMER_MONTHS)    # Jun, Jul, Aug

# Human-readable 30-min slot labels used later for plotting axes
SLOT_LABELS = pd.date_range('00:00', periods=48, freq='30min').strftime('%H:%M')
DOW_LABELS  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

print('Temporal features added.')
display(df[['slot','date','dayofweek','month','is_weekend','is_winter','is_summer']].head(3))

---
## 3 · Feature Engineering

We engineer three complementary feature families:

| Family | Description | # Features |
|--------|-------------|------------|
| **Statistical** | Mean, std, CV, percentiles, zero-days fraction | 9 |
| **Temporal** | Winter/summer ratio, weekday/weekend ratio, peak-hour share, autocorr | 8 |
| **Fourier — daily profile** | FFT harmonics H1–H8 of the 48-slot average daily curve | 8 |
| **Fourier — inter-day** | FFT amplitudes at weekly, bi-weekly, monthly, quarterly, annual frequencies | 5 |

**Total: ~30 features per customer before PCA.**

In [ ]:
# ── 3a  Statistical + temporal aggregation features ──────────────────────────
def _stats_and_temporal(group: pd.DataFrame) -> pd.Series:
    """One row of features per customer (statistical + temporal)."""
    val       = group['energy_kwh']
    daily_tot = group.groupby('date')['energy_kwh'].sum()
    n_days    = len(daily_tot)

    # ── Statistical
    mean_v  = val.mean()
    std_v   = val.std()
    feats = {
        'mean_kwh'        : mean_v,
        'std_kwh'         : std_v,
        'cv_kwh'          : std_v / mean_v        if mean_v > 0 else 0.0,
        'max_kwh'         : val.max(),
        'p25_kwh'         : val.quantile(0.25),
        'p75_kwh'         : val.quantile(0.75),
        'p90_kwh'         : val.quantile(0.90),
        # Fraction of days with near-zero consumption  ← key RS indicator
        'zero_days_frac'  : (daily_tot < 0.01).sum() / n_days if n_days > 0 else 0.0,
        'low_days_frac'   : (daily_tot < daily_tot.quantile(0.1)).sum() / n_days if n_days > 1 else 0.0,
    }

    # ── Seasonal
    winter_mask = group['is_winter']
    summer_mask = group['is_summer']
    w_mean = val[winter_mask].mean() if winter_mask.any() else 0.0
    s_mean = val[summer_mask].mean() if summer_mask.any() else 0.0
    feats['winter_mean']         = w_mean
    feats['summer_mean']         = s_mean
    feats['winter_summer_ratio'] = w_mean / s_mean if s_mean > 0 else 10.0

    # ── Weekly
    wkday_mask = ~group['is_weekend']
    wkend_mask = group['is_weekend']
    wd_mean = val[wkday_mask].mean() if wkday_mask.any() else 0.0
    we_mean = val[wkend_mask].mean() if wkend_mask.any() else 0.0
    feats['weekday_mean']         = wd_mean
    feats['weekend_mean']         = we_mean
    feats['weekend_weekday_ratio']= we_mean / wd_mean if wd_mean > 0 else 1.0

    # ── Behavioural
    feats['peak_share']    = val[group['hour'].between(17, 21)].sum() / val.sum() if val.sum() > 0 else 0.0
    feats['daily_autocorr']= daily_tot.autocorr(lag=1) if len(daily_tot) > 2 else 0.0

    return pd.Series(feats)


COLS_FOR_STATS = ['energy_kwh', 'date', 'is_winter', 'is_summer', 'is_weekend', 'hour']
print('Computing statistical & temporal features … (~30 s for 8.7 M rows)')
stats_feats = (
    df.groupby(METER_ID_COL)[COLS_FOR_STATS]
    .apply(_stats_and_temporal, include_groups=False)
    .reset_index()
)
stats_feats['daily_autocorr'] = stats_feats['daily_autocorr'].fillna(0.0)
print(f'Stats/temporal features shape: {stats_feats.shape}')
display(stats_feats.head(3).T)

---
## 4 · Fourier Decomposition

### 4a — Intra-day profile shape (FFT of the 48-slot daily average)

Each customer's **average daily curve** (48 half-hourly kWh values) captures their
typical consumption rhythm over 24 h.  
Applying the DFT to this 48-point signal reveals:

- **H1** (1 cycle / day): broad daily wave — overall 24 h rise and fall  
- **H2** (2 cycles / day): **double-peak** structure (morning + evening) — characteristic of RP with electric heating  
- **H3–H8**: finer intra-day shape features

All harmonic amplitudes are normalised by the DC component (H0 = mean) so features
are scale-invariant across customers.

### 4b — Inter-day periodicity (FFT of the 364-day daily-totals series)

For a time series of 364 daily totals (= 52 complete weeks), the DFT index
maps directly to calendar periodicities:

| DFT index | Period | Significance for RP/RS |
|-----------|--------|------------------------|
| 1 | Annual (364 d) | Seasonal swing — present for both |
| 4 | Quarterly (91 d) | Spring/autumn transitions |
| 12 | Monthly (≈30 d) | Monthly billing cycles |
| 26 | Bi-weekly (14 d) | Alternating weekend visits — elevated for RS |
| **52** | **Weekly (7 d)** | **Key RS discriminator**: strong for vacation homes that fill on weekends |

In [ ]:
# ── 4a  Fourier of the average daily profile (500 × 48 matrix) ───────────────
profile_matrix = (
    df.groupby([METER_ID_COL, 'slot'])['energy_kwh']
    .mean()
    .unstack(fill_value=0.0)   # shape (500, 48)
)
print(f'Daily profile matrix : {profile_matrix.shape}')

N_HARMONICS = 8   # H1 through H8

def _profile_fft(row_48: np.ndarray) -> dict:
    """Normalised FFT amplitudes H1–H8 for a 48-point daily profile."""
    amps = np.abs(rfft(row_48))      # shape (25,): H0, H1, …, H24
    dc   = amps[0]
    return {
        f'profile_fft_h{k}': float(amps[k] / dc) if dc > 0 else 0.0
        for k in range(1, N_HARMONICS + 1)
    }

profile_fft = (
    pd.DataFrame(
        [_profile_fft(row) for row in profile_matrix.values],
        index=profile_matrix.index,
    )
    .reset_index()                   # 'id' column + 8 FFT columns
)
print(f'Profile FFT features : {profile_fft.shape}')
display(profile_fft.head(3))

In [ ]:
# ── 4b  Fourier of the daily-totals time series (364-day series per customer) ─
N_DAYS = 364   # 52 complete weeks — exact for this dataset

def _daily_totals_fft(group: pd.DataFrame) -> pd.Series:
    """FFT amplitudes at key periodicities from the 364-day daily-total series."""
    daily_series = group.groupby('date')['energy_kwh'].sum()

    # Build a gapless 364-day index starting from the first recorded date
    date_index = pd.date_range(
        start=pd.Timestamp(daily_series.index.min()),
        periods=N_DAYS,
        freq='D',
    ).date
    totals = daily_series.reindex(date_index).fillna(0.0).values

    spectrum = np.abs(rfft(totals))      # shape (183,)
    dc       = spectrum[0]
    norm     = spectrum / dc if dc > 0 else spectrum

    # Map named periodicities to their DFT frequency index
    key_indices = {
        'fft_annual'   : 1,
        'fft_quarterly': int(round(N_DAYS / 91)),
        'fft_monthly'  : int(round(N_DAYS / 30)),
        'fft_biweekly' : int(round(N_DAYS / 14)),
        'fft_weekly'   : int(round(N_DAYS / 7)),   # = 52  ← RS discriminator
    }
    return pd.Series({
        name: float(norm[idx]) if idx < len(norm) else 0.0
        for name, idx in key_indices.items()
    })


print('Computing inter-day FFT features … (~20 s)')
daily_fft = (
    df.groupby(METER_ID_COL)[['date', 'energy_kwh']]
    .apply(_daily_totals_fft, include_groups=False)
    .reset_index()
)
print(f'Inter-day FFT features : {daily_fft.shape}')
display(daily_fft.head(3))

In [ ]:
# ── 4c  Assemble complete feature matrix ──────────────────────────────────────
features = (
    stats_feats
    .merge(profile_fft, on=METER_ID_COL)
    .merge(daily_fft,   on=METER_ID_COL)
)

FEATURE_COLS = [c for c in features.columns if c != METER_ID_COL]

print(f'Full feature matrix : {features.shape}  ({len(FEATURE_COLS)} features)')
print(f'Features : {FEATURE_COLS}')
display(features[FEATURE_COLS].describe().round(3))

---
## 5 · PCA Dimensionality Reduction

Before feeding into K-Means, we:
1. **Standardize** all features (zero mean, unit variance) so that high-magnitude features
   (e.g., `max_kwh`) don't dominate the Euclidean distance used by K-Means.
2. **PCA** — keep principal components that together explain ≥ 90 % of the total variance.
   This removes noise dimensions and helps K-Means converge to a better solution.

In [ ]:
X_raw    = features[FEATURE_COLS].fillna(0.0).values    # (500, ~30)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

pca      = PCA(n_components=0.90, random_state=42)
X_pca    = pca.fit_transform(X_scaled)

print(f'Input shape   : {X_scaled.shape}')
print(f'PCA output    : {X_pca.shape}  ({pca.n_components_} components)')
print(f'Total variance explained : {pca.explained_variance_ratio_.sum()*100:.1f}%')

# ── Scree plot ─────────────────────────────────────────────────────────────
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
indvar = pca.explained_variance_ratio_ * 100
labels_pc = [f'PC{i+1}' for i in range(pca.n_components_)]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Variance per Component (%)', 'Cumulative Variance Explained (%)'],
)
fig.add_trace(go.Bar(x=labels_pc, y=indvar, marker_color='steelblue', name='Individual'), row=1, col=1)
fig.add_trace(go.Scatter(x=labels_pc, y=cumvar, mode='lines+markers',
                         line=dict(color='tomato', width=2), marker=dict(size=7), name='Cumulative'), row=1, col=2)
fig.add_hline(y=90, line_dash='dash', line_color='grey', annotation_text='90 %', row=1, col=2)
fig.update_layout(title='PCA Scree Plot', height=380, template='plotly_white', showlegend=False)
fig.show()

---
## 6 · K-Means Clustering

### 6a — Choosing k: Elbow + Silhouette

- **Elbow curve (WCSS)**: we look for the inflection point where additional clusters yield
  diminishing returns on intra-cluster compactness.  
- **Silhouette score**: measures how well each point fits its own cluster vs the nearest
  neighbour cluster — higher is better (max = 1.0).  

We expect the domain knowledge (binary RP / RS) to be confirmed by a clear elbow at k = 2.

In [ ]:
K_RANGE   = range(2, 11)
wcss_vals = []
sil_vals  = []

print('Running K-Means for k = 2 … 10 …')
for k in K_RANGE:
    km  = KMeans(n_clusters=k, random_state=42, n_init=20)
    lbl = km.fit_predict(X_pca)
    wcss_vals.append(km.inertia_)
    sil_vals.append(silhouette_score(X_pca, lbl))
    print(f'  k={k}  WCSS={km.inertia_:>10,.1f}  silhouette={sil_vals[-1]:.4f}')

# ── Dual-panel chart ───────────────────────────────────────────────────────
ks = list(K_RANGE)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Elbow Curve — WCSS (lower = tighter clusters)',
                    'Silhouette Score (higher = better separation)'],
)
fig.add_trace(go.Scatter(x=ks, y=wcss_vals, mode='lines+markers',
                         line=dict(color='steelblue', width=2.5),
                         marker=dict(size=9)), row=1, col=1)
fig.add_trace(go.Scatter(x=ks, y=sil_vals, mode='lines+markers',
                         line=dict(color='tomato', width=2.5),
                         marker=dict(size=9)), row=1, col=2)

# Annotate the chosen k=2
for col in (1, 2):
    fig.add_vline(x=2, line_dash='dash', line_color='green',
                  annotation_text=' k=2 selected', annotation_position='top right',
                  row=1, col=col)

fig.update_xaxes(title_text='Number of clusters k', dtick=1)
fig.update_yaxes(title_text='WCSS', row=1, col=1)
fig.update_yaxes(title_text='Silhouette', row=1, col=2)
fig.update_layout(title='K-Means Cluster Selection', height=420,
                  template='plotly_white', showlegend=False)
fig.show()

### 6b — Fit K-Means with k = 2 and assign RP / RS labels

**Label assignment rule (data-driven):**  
Electric-heating primary residences consume significantly more energy in winter
(heating season) than in summer.  
→ The cluster with the **higher mean `winter_summer_ratio`** is labelled **RP**.
The other cluster, characterised by lower overall consumption, more absent days, and
a higher weekly FFT amplitude (irregular weekend visits), is labelled **RS**.

In [ ]:
K = 2
kmeans      = KMeans(n_clusters=K, random_state=42, n_init=20)
cluster_ids = kmeans.fit_predict(X_pca)

features['cluster'] = cluster_ids

# ── Determine which cluster is RP using domain knowledge ──────────────────
discriminators = ['winter_summer_ratio', 'zero_days_frac', 'mean_kwh', 'fft_weekly']
cluster_stats  = features.groupby('cluster')[discriminators].mean()
print('Cluster mean values for label assignment:')
display(cluster_stats.round(4))

rp_cluster = int(cluster_stats['winter_summer_ratio'].idxmax())  # higher ratio → RP
rs_cluster = 1 - rp_cluster

features['label'] = features['cluster'].map({rp_cluster: 'RP', rs_cluster: 'RS'})
# Binary integer label for downstream supervised models  (0 = RP, 1 = RS)
features['residence_type_kmeans'] = features['cluster'].map({rp_cluster: 0, rs_cluster: 1})

counts = features['label'].value_counts()
print(f'\nCluster {rp_cluster} → RP  |  Cluster {rs_cluster} → RS')
print(f'RP : {counts.get("RP", 0):>4} customers  ({counts.get("RP", 0)/len(features)*100:.1f}%)')
print(f'RS : {counts.get("RS", 0):>4} customers  ({counts.get("RS", 0)/len(features)*100:.1f}%)')

In [ ]:
# ── Cluster validation summary ─────────────────────────────────────────────
SUMMARY_COLS = [
    'mean_kwh', 'winter_mean', 'summer_mean', 'winter_summer_ratio',
    'zero_days_frac', 'weekend_weekday_ratio', 'fft_weekly', 'peak_share',
    'profile_fft_h2',   # double-peak amplitude — higher for RP (morning + evening)
]
summary = (
    features.groupby('label')[SUMMARY_COLS]
    .mean()
    .round(4)
    .T
    .rename_axis('Feature')
)
summary.columns.name = 'Cluster'
print('Full cluster summary (mean values):')
display(summary)

---
## 7 · Save the Labeled Dataset

In [ ]:
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = DATA_PROCESSED_DIR / 'labeled_customers.parquet'
features.to_parquet(output_path, index=False)
print(f'Labeled dataset saved → {output_path}')
print(f'Shape : {features.shape}')
print(f'Columns: {features.columns.tolist()}')

# Attach labels back onto the raw time series for downstream visualisation
df = df.merge(features[[METER_ID_COL, 'label']], on=METER_ID_COL, how='left')
print('\nRaw time series with labels:')
display(df[['id','horodate','energy_kwh','label']].sample(5, random_state=1))

---
## 8 · Validation Visualisations

Three Plotly charts to validate the clustering result visually:

1. **Average 24-hour profile** — the main validation plot  
2. **Seasonal decomposition** — winter vs summer by cluster  
3. **Day-of-week profile** — weekday vs weekend patterns by cluster

In [ ]:
# ── Chart 1: Average daily consumption profile  RP vs RS  ─────────────────
COLORS = {'RP': '#1F77B4', 'RS': '#FF7F0E'}   # blue / orange

avg_profile = (
    df.groupby(['label', 'slot'])['energy_kwh']
    .mean()
    .reset_index()
    .pivot(index='slot', columns='label', values='energy_kwh')
)

fig = go.Figure()
for lbl in ['RP', 'RS']:
    if lbl in avg_profile.columns:
        fig.add_trace(go.Scatter(
            x=SLOT_LABELS,
            y=avg_profile[lbl].values,
            mode='lines',
            name=lbl,
            line=dict(color=COLORS[lbl], width=3),
            hovertemplate='%{x}<br>%{y:.4f} kWh<extra>' + lbl + '</extra>',
        ))

# Annotate characteristic time windows
fig.add_vrect(x0='06:30', x1='09:00', fillcolor='lightblue',  opacity=0.15,
              annotation_text='Morning peak', annotation_position='top left')
fig.add_vrect(x0='17:00', x1='22:00', fillcolor='lightyellow', opacity=0.20,
              annotation_text='Evening peak', annotation_position='top left')

fig.update_layout(
    title={
        'text': 'Average 30-min Consumption Profile — Primary (RP) vs Secondary (RS) Residences',
        'font': {'size': 16},
    },
    xaxis_title='Time of Day (30-min slots)',
    yaxis_title='Mean Energy (kWh / 30 min)',
    xaxis=dict(tickangle=-45, dtick=4),
    legend=dict(title='Residence Type', font=dict(size=13)),
    height=520,
    template='plotly_white',
    hovermode='x unified',
)
fig.show()

In [ ]:
# ── Chart 2: Seasonal profiles — Winter vs Summer, RP vs RS ───────────────
seasonal = (
    df.groupby(['label', 'is_winter', 'is_summer', 'slot'])['energy_kwh']
    .mean()
    .reset_index()
)
winter_df = seasonal[seasonal['is_winter']].sort_values('slot')
summer_df = seasonal[seasonal['is_summer']].sort_values('slot')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Winter (Nov – Mar)', 'Summer (Jun – Aug)'],
    shared_yaxes=True,
)
for col_idx, (season_df, season_name) in enumerate([(winter_df, 'Winter'), (summer_df, 'Summer')], 1):
    for lbl in ['RP', 'RS']:
        sub = season_df[season_df['label'] == lbl]
        fig.add_trace(
            go.Scatter(
                x=SLOT_LABELS,
                y=sub['energy_kwh'].values,
                mode='lines',
                name=f'{lbl} ({season_name})',
                line=dict(color=COLORS[lbl], width=2.5,
                          dash='solid' if season_name == 'Winter' else 'dot'),
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )

fig.update_xaxes(title_text='Time of Day', tickangle=-45, dtick=4)
fig.update_yaxes(title_text='Mean Energy (kWh / 30 min)', row=1, col=1)
fig.update_layout(
    title='Seasonal Consumption Profiles — RP vs RS',
    height=500,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(title='Cluster × Season', font=dict(size=12)),
)
fig.show()

In [ ]:
# ── Chart 3: Day-of-week mean consumption — RP vs RS ──────────────────────
weekly_profile = (
    df.groupby(['label', 'dayofweek'])['energy_kwh']
    .mean()
    .reset_index()
    .pivot(index='dayofweek', columns='label', values='energy_kwh')
    .reindex(range(7))
)

fig = go.Figure()
for lbl in ['RP', 'RS']:
    if lbl in weekly_profile.columns:
        fig.add_trace(go.Bar(
            x=DOW_LABELS,
            y=weekly_profile[lbl].values,
            name=lbl,
            marker_color=COLORS[lbl],
            opacity=0.85,
            hovertemplate='%{x}<br>%{y:.4f} kWh<extra>' + lbl + '</extra>',
        ))

# Highlight weekend columns
fig.add_vrect(x0=4.5, x1=6.5, fillcolor='lightyellow', opacity=0.3,
              annotation_text='Weekend', annotation_position='top left')

fig.update_layout(
    title='Average Daily Energy by Day of Week — RP vs RS',
    xaxis_title='Day of Week',
    yaxis_title='Mean Energy (kWh / 30 min)',
    barmode='group',
    height=440,
    template='plotly_white',
    legend=dict(title='Residence Type', font=dict(size=13)),
)
fig.show()

---
## 9 · Summary & What to Expect in the Plots

### Expected RP curve (Primary Residence)
- **Higher overall level** — people are always home → continuous base load + heating
- **Pronounced morning and evening peaks** — wake-up + return from work heating cycles
- **High winter amplitude** — electric heating responds to outdoor temperature
- **Consistent across weekdays** — occupants follow a regular schedule

### Expected RS curve (Secondary Residence)
- **Lower mean consumption** — property is unoccupied for long stretches
- **Flat or near-zero profile** on weekdays (absence)
- **Weekend spikes** — Friday evening arrival, Sunday departure — visible in day-of-week chart
- **Higher weekly FFT amplitude** — consumption is strongly periodic at the 7-day scale
- **Lower `winter_summer_ratio`** — vacation homes in warm regions, or simply less heating

---

### Output artefact

The file **`data/processed/labeled_customers.parquet`** contains:
- One row per customer (500 rows)
- All engineered features (~30 columns)
- `cluster` (0 or 1)
- **`label`** (`'RP'` or `'RS'`) ← used as the ground truth for Step 2 (Classification)
- `residence_type_kmeans` (0 = RP, 1 = RS) ← binary integer for scikit-learn

### Next Steps
| Step | Task |
|------|------|
| **2** | Train a **Logistic Regression** and a **PyTorch Neural Network** as supervised classifiers on these k-means labels |
| **3** | **Forecasting** — Linear Regression, ARIMA, PyTorch RNN/LSTM on daily consumption |
| **4** | **Generation** — Conditional time-series synthesis (VAE / TimeGAN conditioned on RP vs RS) |